# Inference Optimization — vLLM Experiment

## 1. Context and Objective

The previous analysis estimated the GPU cost of using a large VLM to automatically annotate the 10,000 FATURA invoices. Qwen2.5-VL-32B was identified as a candidate teacher model, and an initial H100-based runtime and cost range was estimated.

However, the 32B model has **not yet been run**. Before renting a high-VRAM GPU for the full annotation process, an inference optimization experiment was conducted using **Qwen2.5-VL-7B-Instruct on a Google Colab NVIDIA T4**.

The objective was to compare standard Transformers inference with a **vLLM-based inference server**, while testing concurrency and quantization to improve throughput.

## 2. vLLM

[vLLM](https://github.com/vllm-project/vllm) is an inference and serving framework optimized for large language models. Unlike a basic sequential Transformers implementation, vLLM is designed to efficiently handle multiple requests concurrently and manage GPU memory more efficiently.

One of its key mechanisms is **PagedAttention**, which improves the management of the KV cache and allows GPU memory to be used more efficiently across concurrent requests.

This is particularly relevant to invoice annotation because each document is an independent inference request. Multiple documents can therefore be processed concurrently rather than waiting for each previous request to finish.

## 3. Experimental Setup

Both experiments used:

* **Qwen2.5-VL-7B-Instruct**
* The **same prompt**
* The **same type of document inputs**
* The same NVIDIA T4 GPU

The vLLM configuration additionally used:

* **Concurrency: 6**
* **AWQ quantization**

AWQ (Activation-aware Weight Quantization) reduces the model's memory footprint by using lower-precision weights while aiming to preserve model quality. This leaves more GPU memory available for inference and concurrent requests.

The comparison therefore evaluates the practical performance of the optimized pipeline rather than isolating vLLM alone.

In [1]:
!pip install -q -U transformers accelerate bitsandbytes qwen-vl-utils pillow openai vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111

In [2]:
from google.colab import files

uploaded = files.upload()

Saving fatura_sample_200.zip to fatura_sample_200.zip


In [3]:
!unzip -q fatura_sample_200.zip

In [4]:
!ls images | head

Template10_Instance110.jpg
Template10_Instance146.jpg
Template10_Instance170.jpg
Template10_Instance78.jpg
Template11_Instance103.jpg
Template11_Instance148.jpg
Template11_Instance22.jpg
Template11_Instance58.jpg
Template12_Instance129.jpg
Template12_Instance149.jpg


In [5]:
from google.colab import files

files.upload()

Saving Qwen_annotate_vllm.py to Qwen_annotate_vllm.py


{'Qwen_annotate_vllm.py': b'import argparse\r\nimport asyncio\r\nimport base64\r\nimport csv\r\nimport io\r\nimport json\r\nimport re\r\nimport sys\r\nimport time\r\nfrom pathlib import Path\r\n\r\nfrom openai import AsyncOpenAI\r\nfrom PIL import Image\r\n\r\n# MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"\r\nMODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct-AWQ"\r\n\r\n# Fixed schema shown to the model. This text never changes between invoices,\r\n# so prompt length is constant across the whole run -- only the image varies.\r\nTARGET_SCHEMA = {\r\n    "fields": {\r\n        "date": "...", "taxes": {"items": []},\r\n        "locale": {"country": "...", "currency": "...", "language": "..."},\r\n        "due_date": "...", "po_number": "...", "total_net": "...", "total_tax": "...",\r\n        "line_items": [{\r\n            "quantity": "...", "tax_rate": "...", "tax_amount": "...", "unit_price": "...",\r\n            "description": "...", "total_price": "...", "product_code": "...", "unit_measure": "..

In [6]:
!ls -l Qwen_annotate_vllm.py

-rw-r--r-- 1 root root 10940 Sep 10 13:36 Qwen_annotate_vllm.py


In [7]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [8]:
!pip install vllm-bnb-plugin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.2 MB/s eta 0:00:00


In [17]:
!nohup vllm serve Qwen/Qwen2.5-VL-7B-Instruct-AWQ \
    --dtype float16 \
    --limit-mm-per-prompt '{"image": 1}' \
    --max-model-len 8192 \
    > vllm.log 2>&1 &

In [22]:
import requests

try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    print("Status:", r.status_code)
    print("Response:", r.text)
except Exception as e:
    print("ERROR:", repr(e))

Status: 200
Response: 


In [23]:
!head -n 21 manifest.csv > manifest_20.csv
!python -u Qwen_annotate_vllm.py \
    --images-dir ./images \
    --output-dir ./qwen_annotations_test20 \
    --manifest ./manifest_20.csv

[info] 20 images to annotate. Fixed prompt length: 4487 chars.
[info] vLLM server at http://localhost:8000/v1 (served model: Qwen/Qwen2.5-VL-7B-Instruct-AWQ), concurrency=6
[info] 10/20 done (110s elapsed)
[info] 20/20 done (195s elapsed)
[done] 20 succeeded, 0 failed, in 195.0s (0.10 img/s)


In [24]:
import json

with open("qwen_annotations_test20/Template10_Instance110.json") as f:
    data = json.load(f)

print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "fields": {
    "date": "04-Jul-2007",
    "taxes": {
      "items": [
        {
          "quantity": "3.00",
          "tax_rate": "0.00",
          "tax_amount": "$264.63",
          "unit_price": "$88.21",
          "description": "Stock down since.",
          "total_price": "$264.63",
          "product_code": "",
          "unit_measure": ""
        },
        {
          "quantity": "3.00",
          "tax_rate": "0.00",
          "tax_amount": "$154.38",
          "unit_price": "$51.46",
          "description": "Particular choice.",
          "total_price": "$154.38",
          "product_code": "",
          "unit_measure": ""
        },
        {
          "quantity": "1.00",
          "tax_rate": "0.00",
          "tax_amount": "$87.42",
          "unit_price": "$87.42",
          "description": "Long.",
          "total_price": "$87.42",
          "product_code": "",
          "unit_measure": ""
        },
        {
          "quantity": "4.00",
          "tax_rate": "0.

## 4. Results

| Configuration             | Documents | Success | Failed |  Time |   Throughput |
| ------------------------- | --------: | ------: | -----: | ----: | -----------: |
| Transformers              |        10 |      10 |      0 | 811 s | ~0.012 img/s |
| vLLM + AWQ, concurrency 6 |        20 |      20 |      0 | 195 s | ~0.103 img/s |

The standard Transformers implementation processed 10 documents in **811 seconds**, while the vLLM configuration processed 20 documents in **195 seconds**.

This corresponds to an observed throughput improvement of approximately **8.3×**.

Both configurations successfully produced parseable outputs for all tested documents.

## 5. Interpretation

The experiment shows that the optimized inference pipeline can substantially improve the throughput of large-scale document annotation.

However, the **8.3× improvement should not be attributed to vLLM alone**, since the configurations also differ in concurrency and quantization. The result should instead be interpreted as the performance of the combined:

**vLLM + concurrency 6 + AWQ**

configuration compared with sequential Transformers inference.

The result is nevertheless highly relevant to the planned 10,000-document annotation workload, where processing documents concurrently can significantly reduce GPU rental time.

## 6. Next Step

The next step is to reproduce this benchmark using the planned **Qwen2.5-VL-32B teacher model** on a rented high-VRAM GPU.

The benchmark should measure throughput under different concurrency and quantization configurations before launching the complete 10,000-document annotation process.

The measured throughput can then be used to replace the previous theoretical runtime estimates and calculate the actual GPU cost:

**Total cost = (10,000 / documents per second) × GPU price per second**

This provides a more reliable cost estimate than extrapolating from model size alone.

## 7. Preliminary Extrapolation to the 32B Configuration

Before the empirical benchmark described in Chapter 6 becomes available, the vLLM/AWQ baseline above can be used to produce a **provisional, non-final** cost projection for the 32B model. This projection is included here for budgeting purposes only and is expected to be superseded once the benchmark of Chapter 6 is executed.

### 7.1 Why parameter-count scaling alone is insufficient

A naive extrapolation would scale the 7B runtime by the 32B/7B parameter ratio (≈4.6×) directly. This is misleading for two reasons:

1. **Per-document latency is not a single homogeneous cost.** It combines a vision-encoding pass (compute-bound, one forward pass per image) and an autoregressive text-decoding phase (bandwidth-bound, scales with output length). Only the decoding phase is expected to scale with LLM parameter count; the vision encoder is shared across model sizes in the Qwen2.5-VL family and its cost does not grow with the 32B backbone.
2. **Hardware changes simultaneously with model size** in the target deployment (T4 → rented A100/H100), and the two hardware tiers do not accelerate compute-bound and bandwidth-bound workloads by the same factor. Applying a single "GPU speedup" multiplier to the whole per-document time conflates these two effects.

### 7.2 Component-based estimate

Assuming a representative split of the 7B/T4 baseline into ~15% vision encoding and ~85% text decoding (9.75 s/doc → ≈1.46 s vision, ≈8.29 s decoding):

| Step | Basis | Result |
|---|---|---|
| Decode time scaled to 32B, same GPU (T4) | ×4.57 (parameter ratio) | 8.29 s → 37.9 s |
| Vision time scaled to 32B, same GPU (T4) | unchanged (shared encoder) | 1.46 s |
| **Total, 32B on T4** | | **≈39.4 s/doc** |

Scaling each component separately to the candidate rental GPUs, using memory-bandwidth ratio for the decode phase (bandwidth-bound) and an approximate compute-throughput ratio for the vision phase (compute-bound):

| GPU | Vision (scaled) | Decode (scaled) | **Estimated total/doc** |
|---|---:|---:|---:|
| A100 80GB | ≈0.29 s | ≈6.3 s | **≈6.6 s** |
| H100 80GB | ≈0.15 s | ≈3.8 s | **≈3.9 s** |

> *Note on scaling ratios: the decode-phase ratios (A100 ≈6×, H100 ≈10×) are derived from manufacturer-published memory bandwidth specifications (T4 ≈320 GB/s, A100 80GB ≈2 TB/s, H100 80GB ≈3.35 TB/s), since autoregressive decoding at low-to-moderate batch size is typically memory-bandwidth-bound. The vision-phase ratios (A100 ≈5×, H100 ≈10×) are an approximate compute-throughput estimate, since vision encoding is a single compute-bound forward pass per image. These ratios are theoretical (spec-based), not empirically measured on this workload, and do not account for quantization-kernel efficiency differences across GPU architectures or for utilization losses at the tested batch size. They should be treated as a planning approximation pending the empirical benchmark in Chapter 6.*

These figures assume concurrency 6, matching the T4 baseline's VRAM-constrained configuration. A100/H100 offer substantially more VRAM headroom (80GB vs. 16GB), so higher concurrency during the Chapter 6 benchmark may yield further throughput gains beyond this estimate.

### 7.3 Projected cost for the 10,000-document workload

At concurrency 6 (conservative, no additional concurrency benefit assumed) and current on-demand GPU rental rates (~ 1.50dollars/h for A100 80GB, ~ 3.00dollars/h for H100 80GB):

<table>
  <thead>
    <tr>
      <th>GPU</th>
      <th>Estimated time (10,000 docs)</th>
      <th>Estimated cost</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>A100 80GB</td>
      <td>≈18.3 h</td>
      <td>≈$27</td>
    </tr>
    <tr>
      <td>H100 80GB</td>
      <td>≈10.9 h</td>
      <td>≈$33</td>
    </tr>
  </tbody>
</table>

### 7.4 Status of this estimate

These figures should be treated as **upper bounds under a conservative concurrency assumption**, not as a final budget. They are derived from architectural reasoning rather than measurement and depend on assumptions (vision/decode time split, hardware scaling ratios, unchanged output length for the 32B model) that have not been empirically validated. The benchmark described in Chapter 6 is required to confirm or revise these numbers before committing to the full annotation run.
